# AIかわいい判定ラボ

このノートブックでは、動物画像に「かわいい！」「今回はちがう」と教えることで、あなたの好みを学ぶAIを育てます。

AIはみんなに共通の正解を当てるものではありません。あなたが選んだラベルから、あなたの好みに近づこうとします。

## 0. はじめに

**講師が最初に「すべてのセルを実行」で画面を準備します。参加者はボタンだけで進められます。**

1. 3-Aで確認用の写真に回答します（AIには教えない写真です）。
2. 4で学習用に10枚教え、「この枚数で比較・記録」を押します。
3. 「次の段階へ」で累計20枚、次に40枚まで追加し、同じボタンで比較します。
4. 4の「記録を見る」で変化を確認し、最後に「CSV保存」を押します。

回答はPC内に自動保存され、再起動して上から実行しても復元されます。同じPCを次の参加者が使うときは、講師が「新しい参加者」を選び確認してください。セルの再実行では回答を消しません。

40枚で比較した後は、4の「自由に追加して学習する」で残りの学習用写真に回答できます。「AIを更新する」で追加分を学習し、必要なら「この枚数で比較・記録」で比較も残せます。10・20・40枚の記録と確認用回答はそのままです。


## 1. パッケージを準備する

Colabで足りないものがあるときのためのセルです。

In [ ]:
# 【触らない】Colabのみインストール。ローカルは事前に uv sync --locked を実行します。
import sys
import subprocess
if 'google.colab' in sys.modules:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'ipywidgets', 'scikit-learn>=1.7', 'pandas', 'numpy', 'Pillow', 'matplotlib'])
else:
    print('ローカル環境を使います。カーネルは Cute AI (uv) を選んでください。')

## 2. データの場所を決める

In [ ]:
# 【触ってOK】データの場所を指定します。
# Colabに zip を展開した場合、まずはこのままで試してください。
from pathlib import Path
WORKSHOP_DIR = str(next((p for p in [Path('data/workshop'), Path('../data/workshop')] if (p / 'metadata.csv').exists()), Path('data/workshop')))

# Google Driveに置いた場合の例:
# WORKSHOP_DIR = "/content/drive/MyDrive/cute-ai-workshop/data/workshop"

## 3. データを読み込む

In [ ]:
# 【触らない】ライブラリとデータを読み込みます。
import base64
import io
import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from IPython.display import display, clear_output
import ipywidgets as widgets
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

workshop_dir = Path(WORKSHOP_DIR)
metadata_path = workshop_dir / "metadata.csv"
embeddings_path = workshop_dir / "embeddings.npy"
images_dir = workshop_dir / "images"

if not metadata_path.exists():
    raise FileNotFoundError(f"metadata.csv が見つかりません: {metadata_path}\nWORKSHOP_DIR の場所を確認してください。")
if not embeddings_path.exists():
    raise FileNotFoundError(f"embeddings.npy が見つかりません: {embeddings_path}\n講師が事前に embedding を作っているか確認してください。")
if not images_dir.exists():
    raise FileNotFoundError(f"images フォルダが見つかりません: {images_dir}")

metadata = pd.read_csv(metadata_path)
embeddings = np.load(embeddings_path)

if len(metadata) != embeddings.shape[0]:
    raise ValueError("metadata.csv の行数と embeddings.npy の数が合っていません。")

print("読み込みOK")
print(f"画像数: {len(metadata)}")
print(f"embeddingの形: {embeddings.shape}")

## 3-A. 確認用画像に先に回答する

最大20枚を確認用に固定します。AIの予測を見ずに回答してください。この回答はAIには教えず、後であなたの判断との一致率を調べるために使います。

回答が終わったら4の画面へ進みます。セルを再実行しても回答・履歴は保持されます。


In [ ]:
# 【触らない】確認用の画像・回答は学習用と別に管理します。
import hashlib

def make_holdout_mask(frame):
    if len(frame) < 4:
        raise ValueError("4枚以上の画像が必要です。")
    keys = frame["image_path"].astype(str).tolist()
    if len(set(keys)) != len(keys):
        raise ValueError("image_pathが重複しています。講師に確認してください。")
    order = sorted(range(len(keys)), key=lambda i: hashlib.sha256(("holdout-42:" + keys[i]).encode()).hexdigest())
    mask = np.zeros(len(frame), dtype=bool)
    mask[order[:min(20, max(1, len(frame)//5))]] = True
    return mask

import json
import os

# 配布データの外に保存し、配布zipへ参加者の回答を混ぜません。
session_path = workshop_dir.parent.parent / "participant_progress.json"
dataset_key = hashlib.sha256(metadata.to_csv(index=False).encode() + embeddings.tobytes()).hexdigest()
if "_session_dataset_key" in globals() and _session_dataset_key != dataset_key:
    raise ValueError("教材データが変わりました。カーネルを再起動してください。保存済み回答は別の場所へ退避してください。")
holdout_mask = make_holdout_mask(metadata)
if "_session_dataset_key" not in globals():
    restored = None
    if session_path.exists():
        try:
            restored = json.loads(session_path.read_text(encoding="utf-8"))
            if restored["version"] != 1 or restored["dataset_key"] != dataset_key:
                raise ValueError("教材データまたは保存形式が異なります")
            for name in ("labels", "holdout_labels"):
                values = restored[name]
                if len(values) != len(metadata) or any(v not in (-1, 0, 1) for v in values):
                    raise ValueError("回答の形式が異なります")
            if any(restored["labels"][i] != -1 for i in np.flatnonzero(holdout_mask)):
                raise ValueError("確認用画像に学習用回答が含まれています")
            if not isinstance(restored["evaluation_history"], list):
                raise ValueError("履歴の形式が異なります")
        except (ValueError, KeyError, TypeError, OSError) as exc:
            raise ValueError(f"保存データを復元できません。上書きせず講師に確認してください: {session_path} ({exc})") from exc
    labels = np.array(restored["labels"], dtype=int) if restored else np.full(len(metadata), -1, dtype=int)
    holdout_labels = np.array(restored["holdout_labels"], dtype=int) if restored else np.full(len(metadata), -1, dtype=int)
    evaluation_history = restored["evaluation_history"] if restored else []
    completed_stages = restored.get("completed_stages", []) if restored else []
    stage_target = restored.get("stage_target", 10) if restored else 10
    model = None
    scores = np.full(len(metadata), np.nan)
    current_index = None
    rng = random.Random(42)
    _session_dataset_key = dataset_key
    if restored:
        print("保存済みの回答・履歴を復元しました。AIは『AIを更新する』で再学習できます。")

save_status = widgets.HTML()

def persist_progress():
    payload = {"version": 1, "dataset_key": dataset_key,
               "labels": labels.tolist(), "holdout_labels": holdout_labels.tolist(),
               "evaluation_history": evaluation_history,
               "completed_stages": completed_stages, "stage_target": stage_target}
    temporary = session_path.with_suffix(".tmp")
    try:
        temporary.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
        os.replace(temporary, session_path)
        save_status.value = "回答・履歴を自動保存しました。"
        return True
    except OSError:
        save_status.value = "<b>自動保存できません。画面を閉じず、講師に保存先の書き込み権限を確認してもらってください。</b>"
        return False

def holdout_ready():
    return bool(np.all(holdout_labels[holdout_mask] >= 0))

def record_evaluation(name, fitted, train_mask):
    if not holdout_ready():
        raise ValueError("先に確認用画像すべてに回答してください。")
    if np.any(train_mask & holdout_mask):
        raise ValueError("確認用画像が学習に混ざっています。")
    row = {
        "モデル": name, "学習枚数": int(train_mask.sum()),
        "確認枚数": int(holdout_mask.sum()),
        "確認用一致率": fitted.score(embeddings[holdout_mask], holdout_labels[holdout_mask]),
        "学習用一致率": fitted.score(embeddings[train_mask], labels[train_mask]),
        "出題モード": mode_dropdown.value,
        "学習画像No.": ",".join(map(str, np.flatnonzero(train_mask))),
        "確認画像No.": ",".join(map(str, np.flatnonzero(holdout_mask))),
    }
    check_ids = np.flatnonzero(holdout_mask)
    predicted = fitted.predict(embeddings[check_ids])
    probabilities = fitted.predict_proba(embeddings[check_ids])[:, 1]
    row["確認用詳細"] = [
        {"画像No.": int(idx), "自分の回答": int(holdout_labels[idx]),
         "AIの予測": int(pred), "AIかわいい度": float(prob)}
        for idx, pred, prob in zip(check_ids, predicted, probabilities)
    ]
    evaluation_history.append(row)
    persist_progress()
    return row["確認用一致率"]

holdout_out = widgets.Output()
holdout_cute = widgets.Button(description="かわいい！", icon="heart")
holdout_different = widgets.Button(description="今回はちがう", icon="times")

def show_holdout():
    with holdout_out:
        clear_output(wait=True)
        remaining = np.flatnonzero(holdout_mask & (holdout_labels == -1))
        if not len(remaining):
            holdout_cute.disabled = holdout_different.disabled = True
            print("確認用への回答が完了しました。次のセルで学習を始めましょう。")
            if len(np.unique(holdout_labels[holdout_mask])) == 1:
                print("確認用の回答は1種類だけです。一致率が高くても両方を分けられるとは限りません。")
            return
        idx = int(remaining[0])
        with Image.open(workshop_dir / metadata.iloc[idx]["image_path"]) as source:
            picture = source.convert("RGB")
        picture.thumbnail((360, 360))
        display(picture)
        print(f"確認用 No.{idx} / 残り{len(remaining)}枚。AIの予測を見ずに回答してください。")

def answer_holdout(value):
    remaining = np.flatnonzero(holdout_mask & (holdout_labels == -1))
    if len(remaining):
        holdout_labels[remaining[0]] = value
        persist_progress()
    show_holdout()
    if "show_current" in globals():
        show_current()

holdout_cute.on_click(lambda _: answer_holdout(1))
holdout_different.on_click(lambda _: answer_holdout(0))
display(widgets.VBox([save_status, holdout_out, widgets.HBox([holdout_cute, holdout_different])]))
show_holdout()


## 4. ラベルを付けてAIを育てる

「かわいい！」「今回はちがう」「スキップ」で回答します。

**10枚教える → この枚数で比較・記録 → 結果を見る → 次の段階へ → 累計20枚 → 累計40枚**

各段階で回答が止まり、比較を案内します。両方の回答が必要です。片方だけの場合は追加で教え、実際の枚数で比較します。「AIの推しTOP10」で推薦も見てみましょう。回答・履歴は自動保存されます。

40枚で比較した後は、4の「自由に追加して学習する」で残りの学習用写真に回答できます。「AIを更新する」で追加分を学習し、必要なら「この枚数で比較・記録」で比較も残せます。10・20・40枚の記録と確認用回答はそのままです。


In [ ]:
# 【触らない】かわいい判定ラボの画面を作ります。
# ラベル: -1 = まだ, 0 = 今回はちがう, 1 = かわいい
# 状態は3-Aで準備・復元します。画面を再作成しても初期化しません。
for button, callback in globals().get("_map_callbacks", []):
    button.on_click(callback, remove=True)
_map_callbacks = []
if "app" in globals():
    app.close()

image_out = widgets.Output()
status_out = widgets.Output()
top10_out = widgets.Output()
message = widgets.HTML(value="")

mode_dropdown = widgets.Dropdown(
    options=[("自動", "auto"), ("ランダム", "random"), ("AIが迷っている画像", "uncertain")],
    value="random",
    disabled=True,
    description="出題モード:",
    style={"description_width": "initial"},
)

cute_button = widgets.Button(description="かわいい！", button_style="success", icon="heart")
different_button = widgets.Button(description="今回はちがう", button_style="warning", icon="times")
skip_button = widgets.Button(description="スキップ", button_style="", icon="step-forward")
train_button = widgets.Button(description="AIを更新する", button_style="primary", icon="refresh")
top10_button = widgets.Button(description="AIの推しTOP10", button_style="info", icon="star")
reset_button = widgets.Button(description="ラベルをリセット", button_style="danger", icon="trash")
save_button = widgets.Button(description="CSV保存", button_style="", icon="save")


def image_path(i):
    return workshop_dir / metadata.iloc[i]["image_path"]


def labeled_counts():
    labeled = int(np.sum(labels != -1))
    cute = int(np.sum(labels == 1))
    different = int(np.sum(labels == 0))
    return labeled, cute, different


def update_status():
    labeled, cute, different = labeled_counts()
    with status_out:
        clear_output(wait=True)
        trained = "学習済み" if model is not None else "まだ学習していません"
        print(f"ラベル済み: {labeled}枚 / かわいい！: {cute}枚 / 今回はちがう: {different}枚 / AI: {trained}")

    if not holdout_ready():
        stage_guide.value = "まず3-Aの確認用画像すべてに回答してください。"
    elif stage_target is None:
        stage_guide.value = "<b>自由に追加学習できます。</b>回答を追加して『AIを更新する』を押しましょう。『この枚数で比較・記録』も使えます。" if current_index is not None else "学習用の写真すべてに回答しました。AIを更新し、記録を見て振り返りましょう。"
    elif labeled >= stage_target and min(cute, different) == 0:
        stage_guide.value = "両方の回答が必要です。もう少し写真を見て、自分の好みに沿って回答してください。実際の枚数で比較します。"
    elif labeled >= stage_target:
        stage_guide.value = f"<b>{labeled}枚まで回答しました。</b>『この枚数で比較・記録』を押して結果を見ましょう。"
    else:
        stage_guide.value = f"次は累計<b>{stage_target}枚</b>。あと{stage_target-labeled}枚教えましょう。"
    next_stage_button.description = "自由に追加して学習する" if stage_target == 40 else "次の段階へ"
    next_stage_button.layout.width = "210px"
    next_stage_button.disabled = stage_target is None or stage_target not in completed_stages
    for button in (cute_button, different_button, skip_button):
        button.disabled = not holdout_ready() or current_index is None or (stage_target is not None and labeled >= stage_target and min(cute, different) > 0)


def choose_next_index(exclude=None):
    if not holdout_ready():
        return None
    unlabeled = np.where((labels == -1) & ~holdout_mask)[0]
    if exclude is not None and len(unlabeled) > 1:
        unlabeled = unlabeled[unlabeled != exclude]
    if len(unlabeled) == 0:
        return None

    mode = mode_dropdown.value
    if mode == "auto":
        mode = "uncertain" if model is not None else "random"

    if mode == "uncertain" and model is not None:
        uncertainty = np.abs(scores[unlabeled] - 0.5)
        order = np.argsort(uncertainty)
        candidates = unlabeled[order[: min(10, len(order))]]
        return int(rng.choice(list(candidates)))

    return int(rng.choice(list(unlabeled)))


def show_current():
    global current_index
    if current_index is None:
        current_index = choose_next_index()

    with image_out:
        clear_output(wait=True)
        if current_index is None:
            print("学習用画像の回答が完了しました。" if holdout_ready() else "先に3-Aの確認用画像すべてに回答してください。")
            update_status()
            return

        row = metadata.iloc[current_index]
        img = Image.open(image_path(current_index)).convert("RGB")
        img.thumbnail((420, 420))
        display(img)
        print(f"No.{current_index}  species: {row.get('species', '?')} / breed: {row.get('breed', '?')}")
        if np.isfinite(scores[current_index]):
            print(f"AIかわいい度: {scores[current_index] * 100:.1f}%")
        else:
            print("AIかわいい度: AIを更新すると表示されます")

    update_status()


def set_label(value):
    global current_index
    if current_index is None or holdout_mask[current_index]:
        return
    count, cute, different = labeled_counts()
    if stage_target is not None and count >= stage_target and min(cute, different) > 0:
        return
    labels[current_index] = value
    persist_progress()
    current_index = choose_next_index()
    show_current()


def train_model(_=None):
    global model, scores
    if not holdout_ready():
        message.value = "先に3-Aの確認用画像すべてに回答してください。"
        return
    labeled = (labels != -1) & ~holdout_mask
    y = labels[labeled]

    if np.sum(y == 1) < 1 or np.sum(y == 0) < 1:
        message.value = "<b>まだ学習できません。</b>『かわいい！』と『今回はちがう』を最低1枚ずつ教えてください。"
        update_status()
        return

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)),
    ])
    model.fit(embeddings[labeled], y)
    scores = model.predict_proba(embeddings)[:, 1]
    accuracy = record_evaluation("ふつうのAI", model, labeled)
    message.value = f"<b>AIを更新しました。</b> 学習{labeled.sum()}枚 / 固定確認用一致率: {accuracy:.1%}"
    update_status()
    show_current()


def show_top10(_=None):
    with top10_out:
        clear_output(wait=True)
        if model is None:
            print("先に『AIを更新する』を押してください。")
            return
        pool = np.flatnonzero(~holdout_mask)
        order = pool[np.argsort(-scores[pool])]
        print("AIの推しTOP10")
        shown = 0
        for idx in order:
            row = metadata.iloc[idx]
            img = Image.open(image_path(idx)).convert("RGB")
            img.thumbnail((220, 220))
            display(img)
            label_text = "未ラベル" if labels[idx] == -1 else ("かわいい！" if labels[idx] == 1 else "今回はちがう")
            print(f"No.{idx} / AIかわいい度: {scores[idx] * 100:.1f}% / {label_text} / {row.get('breed', '?')}")
            shown += 1
            if shown >= 10:
                break


def reset_labels(_=None):
    global labels, scores, model, current_index, stage_target
    labels = np.full(len(metadata), -1, dtype=int)
    scores = np.full(len(metadata), np.nan, dtype=float)
    model = None
    current_index = choose_next_index()
    evaluation_history.clear()
    disagreement_choice.options = []
    disagreement_view.children = ()
    completed_stages.clear()
    stage_target = 10
    persist_progress()
    message.value = "学習用ラベルと履歴をリセットしました。確認用画像・回答は維持します。"
    top10_out.clear_output()
    show_current()


def save_labels(_=None):
    labeled, cute, different = labeled_counts()
    result = metadata.copy()
    result["split"] = np.where(holdout_mask, "validation", "train_pool")
    result["validation_label"] = holdout_labels
    result["label"] = labels
    result["label_text"] = np.where(labels == 1, "かわいい！", np.where(labels == 0, "今回はちがう", "未ラベル"))
    result["ai_cute_score"] = scores
    output_path = "my_cute_ai_labels.csv"
    result.to_csv(output_path, index=False)
    pd.DataFrame(evaluation_history).to_csv("my_cute_ai_evaluation_history.csv", index=False)
    message.value = f"CSVを保存しました: <b>{output_path}</b>（ラベル済み {labeled}枚）"


cute_button.on_click(lambda _: set_label(1))
different_button.on_click(lambda _: set_label(0))
skip_button.on_click(lambda _: next_image())
train_button.on_click(train_model)
top10_button.on_click(show_top10)
def request_reset(_=None):
    reset_choice.value = "training"
    reset_confirm.layout.display = ""

reset_button.on_click(request_reset)
save_button.on_click(save_labels)


def next_image():
    global current_index
    current_index = choose_next_index(exclude=current_index)
    show_current()

stage_guide = widgets.HTML()
stage_compare_button = widgets.Button(description="この枚数で比較・記録", button_style="primary", layout=widgets.Layout(width="190px"))
next_stage_button = widgets.Button(description="次の段階へ", disabled=True)
history_button = widgets.Button(description="記録を見る")
stage_out = widgets.Output()
new_participant_button = widgets.Button(description="新しい参加者")
reset_choice = widgets.Dropdown(options=[("学習用だけやり直す", "training"), ("新しい参加者（確認用も消す）", "all")])
confirm_reset_button = widgets.Button(description="保存してからリセット", button_style="danger", layout=widgets.Layout(width="190px"))
cancel_reset_button = widgets.Button(description="キャンセル")
reset_confirm = widgets.VBox([widgets.HTML("回答を消します。現在の自動保存ファイルをバックアップしてから実行します。"), reset_choice,
                             widgets.HBox([confirm_reset_button, cancel_reset_button])], layout=widgets.Layout(display="none"))

# 評価時点の予測を表示し、現在のモデルで再計算しません。
disagreement_button = widgets.Button(description="判断の違う画像を見る", icon="search",
                                    layout=widgets.Layout(width="200px"))
disagreement_choice = widgets.Dropdown(options=[], description="評価記録:",
                                      layout=widgets.Layout(width="100%"))
disagreement_view = widgets.VBox()

def disagreement_rows(record):
    return [item for item in record.get("確認用詳細", [])
            if item["自分の回答"] != item["AIの予測"]]

def render_disagreements(_=None):
    from html import escape
    selection = disagreement_choice.value
    if selection is None or selection >= len(evaluation_history):
        disagreement_view.children = (widgets.HTML("先に『この枚数で比較・記録』を押してください。"),)
        return
    record = evaluation_history[selection]
    details = record.get("確認用詳細", [])
    differences = disagreement_rows(record)
    header = widgets.HTML(
        f"<b>{escape(record['モデル'])} / 学習{record['学習枚数']}枚の記録</b>"
        f"<br>確認用{len(details)}枚中、判断が違った画像は{len(differences)}枚です。"
        "<br>表示は評価した時点の結果です。好みの違いを観察し、確認用の回答は変更しません。"
        "<br>この結果を見て改善した後の一致率は、独立した最終テストの成績ではありません。")
    children = [header]
    if not differences:
        children.append(widgets.HTML("この記録では、確認用の回答とAIの予測がすべて一致しました。"))
    for item in differences:
        idx = item["画像No."]
        answer = "かわいい！" if item["自分の回答"] == 1 else "今回はちがう"
        prediction = "かわいい！" if item["AIの予測"] == 1 else "今回はちがう"
        caption = widgets.HTML(
            f"<b>No.{idx}</b><br>自分の回答: {answer}<br>AIの予測: {prediction}"
            f"<br>AIかわいい度: {item['AIかわいい度']*100:.1f}%")
        try:
            with Image.open(image_path(idx)) as source:
                picture = source.convert("RGB")
            picture.thumbnail((200, 160))
            buffer = io.BytesIO()
            picture.save(buffer, format="PNG")
            thumbnail = widgets.Image(value=buffer.getvalue(), format="png")
        except OSError:
            thumbnail = widgets.HTML("画像を開けません。教材フォルダを確認してください。")
        children.append(widgets.HBox([thumbnail, caption],
                        layout=widgets.Layout(flex_flow="row wrap", margin="8px 0")))
    disagreement_view.children = tuple(children)

def show_disagreements(_=None):
    available = [(f"記録{i+1}: {r['モデル']} / 学習{r['学習枚数']}枚", i)
                 for i, r in enumerate(evaluation_history) if r.get("確認用詳細")]
    disagreement_choice.options = available
    if available:
        disagreement_choice.value = available[-1][1]
        render_disagreements()
    else:
        disagreement_view.children = (widgets.HTML(
            "画像別の予測が保存された記録がありません。『この枚数で比較・記録』を押してください。旧版の一致率の記録はそのまま保持しています。"),)

disagreement_button.on_click(show_disagreements)
disagreement_choice.observe(render_disagreements, names="value")

def show_history(_=None):
    with stage_out:
        clear_output(wait=True)
        if evaluation_history:
            display(pd.DataFrame(evaluation_history).drop(columns=["学習画像No.", "確認画像No.", "確認用詳細"], errors="ignore"))
            print("同じ確認用写真への一致率です。枚数を増やしても必ず上がるとは限りません。")
        else:
            print("まだ記録がありません。10枚教えて比較しましょう。")

def compare_stage(_=None):
    stage_compare_button.disabled = True
    try:
        with stage_out:
            clear_output(wait=True)
            count, cute, different = labeled_counts()
            if not holdout_ready() or min(cute, different) == 0:
                print("確認用への回答を終え、学習用で両方の回答を1枚以上選んでください。")
                return
            if stage_target is not None and count < stage_target:
                print(f"まず累計{stage_target}枚まで教えてください。")
                return
            if "fit_mlp_comparison" not in globals() or "train_ai_with_strength" not in globals():
                print("講師に、最初にすべてのセルを実行してもらってください。回答は保持されます。")
                return
            train_model()
            train_mask = (labels != -1) & ~holdout_mask
            for name, strength in [("慎重なAI", 0.1), ("思い込み強めAI", 1000.0)]:
                fitted, _ = train_ai_with_strength(strength)
                record_evaluation(name, fitted, train_mask)
            comparison = fit_mlp_comparison(embeddings, labels.copy(), holdout_mask, holdout_labels)
            for name, fitted in comparison["models"].items():
                record_evaluation(name, fitted, train_mask)
            display(comparison["summary"])
            if stage_target is not None and stage_target not in completed_stages:
                completed_stages.append(stage_target)
            persist_progress()
            display(pd.DataFrame([row for row in evaluation_history if row["学習枚数"] == count]).drop(columns=["学習画像No.", "確認画像No.", "確認用詳細"], errors="ignore"))
            print("『AIの推しTOP10』や『記録を見る』で確認してください。")
            if stage_target is None:
                print("追加分の比較を記録しました。引き続き回答を追加できます。")
            elif stage_target < 40:
                print("次は『次の段階へ』を押します。")
            else:
                print("40枚の比較が終わりました。『自由に追加して学習する』で続きを試せます。")
    except Exception as exc:
        with stage_out:
            print(f"比較が完了しませんでした。回答は保持しています。講師に確認してください: {exc}")
    finally:
        stage_compare_button.disabled = False
        update_status()
        if "refresh_maps" in globals():
            refresh_maps()

def advance_stage(_=None):
    global stage_target
    if stage_target is None or stage_target not in completed_stages:
        return
    stage_target = {10: 20, 20: 40, 40: None}[stage_target]
    persist_progress()
    show_current()

def request_new_participant(_=None):
    reset_choice.value = "all"
    reset_confirm.layout.display = ""

def confirm_reset(_=None):
    from datetime import datetime
    import shutil
    if not persist_progress():
        return
    try:
        backup = session_path.with_name("participant_progress_backup_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f") + ".json")
        shutil.copy2(session_path, backup)
    except OSError:
        message.value = "バックアップできなかったため、リセットしませんでした。"
        return
    if reset_choice.value == "all":
        holdout_labels[:] = -1
        holdout_cute.disabled = holdout_different.disabled = False
    reset_labels()
    show_holdout()
    stage_out.clear_output()
    reset_confirm.layout.display = "none"
    if "refresh_maps" in globals():
        refresh_maps()

stage_compare_button.on_click(compare_stage)
next_stage_button.on_click(advance_stage)
history_button.on_click(show_history)
new_participant_button.on_click(request_new_participant)
confirm_reset_button.on_click(confirm_reset)
cancel_reset_button.on_click(lambda _: setattr(reset_confirm.layout, "display", "none"))
controls1 = widgets.HBox([cute_button, different_button, skip_button])
controls2 = widgets.HBox([train_button, top10_button, save_button])
app = widgets.VBox([stage_guide, save_status, mode_dropdown, status_out, message, image_out, controls1,
                   widgets.HBox([stage_compare_button, next_stage_button, history_button]), controls2,
                   disagreement_button, disagreement_choice, disagreement_view,
                   stage_out, top10_out, widgets.HBox([reset_button, new_participant_button]), reset_confirm])
display(app)
show_current()

# 4だけを再実行した場合も、既存の地図を新しいボタンにつなぎ直します。
if "refresh_maps" in globals():
    _map_callbacks = [(train_button, refresh_maps), (cute_button, refresh_maps), (different_button, refresh_maps), (map_refresh, refresh_maps)]
    for button, callback in _map_callbacks:
        button.on_click(callback)


## 5. 実験: AIの思い込みを見てみよう

同じラベルを使って、3種類のAIを作ります。

- **慎重なAI**: 少ないラベルだけで強く決めつけにくい
- **ふつうのAI**: いつもの設定
- **思い込み強めAI**: 教えたラベルに強く合わせようとする

少ない例だけを信じすぎると、まだ見ていない画像にも強く決めつけてしまうことがあります。これを機械学習では **過学習** と呼ぶことがあります。


In [ ]:
# 【触らない】同じラベルで、慎重なAI・ふつうのAI・思い込み強めAIを比べます。

def train_ai_with_strength(c_value):
    if not holdout_ready():
        raise ValueError("先に確認用画像すべてに回答してください。")
    labeled = (labels != -1) & ~holdout_mask
    y = labels[labeled]
    if np.sum(y == 1) < 1 or np.sum(y == 0) < 1:
        raise ValueError("『かわいい！』と『今回はちがう』を最低1枚ずつ教えてから実行してください。")
    clf = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(C=c_value, class_weight="balanced", max_iter=1000, random_state=42)),
    ])
    clf.fit(embeddings[labeled], y)
    return clf, clf.predict_proba(embeddings)[:, 1]


def show_ai_comparison(top_n=5):
    settings = [
        ("慎重なAI", 0.1),
        ("ふつうのAI", 1.0),
        ("思い込み強めAI", 1000.0),
    ]
    labeled_count, cute_count, different_count = labeled_counts()
    print(f"使ったラベル: {labeled_count}枚（かわいい！ {cute_count}枚 / 今回はちがう {different_count}枚）")
    print("\nスコアが0%や100%に近いほど、AIは強く決めつけています。\n")

    rows = []
    for name, c_value in settings:
        clf, variant_scores = train_ai_with_strength(c_value)
        unlabeled_scores = variant_scores[(labels == -1) & ~holdout_mask]
        if len(unlabeled_scores) == 0:
            confidence = np.nan
        else:
            confidence = np.mean(np.abs(unlabeled_scores - 0.5)) * 2
        rows.append({
            "AI": name,
            "C": c_value,
            "確認用一致率": record_evaluation(name, clf, (labels != -1) & ~holdout_mask),
            "未ラベル画像への決めつけ強さ": confidence,
            "平均AIかわいい度": np.mean(variant_scores),
        })

    display(pd.DataFrame(rows))

    for name, c_value in settings:
        _, variant_scores = train_ai_with_strength(c_value)
        print("=" * 40)
        print(f"{name} の推しTOP{top_n}")
        for rank, idx in enumerate([i for i in np.argsort(-variant_scores) if not holdout_mask[i]][:top_n], start=1):
            row = metadata.iloc[idx]
            img = Image.open(image_path(idx)).convert("RGB")
            img.thumbnail((180, 180))
            display(img)
            label_text = "未ラベル" if labels[idx] == -1 else ("かわいい！" if labels[idx] == 1 else "今回はちがう")
            print(f"{rank}位: No.{idx} / AIかわいい度 {variant_scores[idx] * 100:.1f}% / {label_text} / {row.get('species', '?')} / {row.get('breed', '?')}")

strength_button = widgets.Button(description="設定による違いを見る")
strength_out = widgets.Output()
def show_strength(_=None):
    with strength_out:
        clear_output(wait=True)
        try:
            show_ai_comparison(top_n=5)
        except ValueError as exc:
            print(exc)
strength_button.on_click(show_strength)
display(widgets.VBox([strength_button, strength_out]))


## 6. 実験: 自分のAIの傾向を見る

AIを更新したあとに実行すると、猫・犬・品種ごとの平均AIかわいい度を見られます。

これは「AIが正しいか」を見るものではありません。自分が付けたラベルから、AIがどんな好みを学んだように見えるかを観察します。


In [ ]:
# 【触らない】species / breed ごとのAIかわいい度を集計します。
if model is None or np.all(np.isnan(scores)):
    print("先に『AIを更新する』を押してください。")
else:
    result = metadata.copy()
    result["ai_cute_score"] = scores
    result["label_text"] = np.where(labels == 1, "かわいい！", np.where(labels == 0, "今回はちがう", "未ラベル"))

    print("species ごとの平均AIかわいい度")
    display(result.groupby("species")["ai_cute_score"].mean().sort_values(ascending=False).to_frame())

    print("\nbreed ごとの平均AIかわいい度 TOP10")
    display(result.groupby("breed")["ai_cute_score"].mean().sort_values(ascending=False).head(10).to_frame())


## 6-A. 実験: 画像の地図とAIの境界を見る

【実行してOK・コードは触らない】このセルは一度実行すれば、上の「AIを更新する」に合わせて図も変わります。「画像No.」を選ぶと、その画像と図の中の位置を確認できます。

- **画像の地図**：似た特徴の画像を近くに並べるt-SNEという方法です。位置は固定して、AIかわいい度で色を付けます。この図には決定境界の線を引きません。
- **判定スコア図**：真ん中の線が境界です。右は「かわいい！」寄り、左は「今回はちがう」寄り。線の近くはAIが迷っています。更新すると線は固定のまま、点の横位置が変わります。
- 図を描くためだけに配置を作っています。判定には元の全特徴量を使います。
- スコアの横軸は `wᵀz + b` です（`z` は標準化した特徴量）。幾何学的な距離そのものではありません。縦軸は固定のランダム配置で、意味はありません。
- AIかわいい度はモデルの予測値です。「正解する確率」ではありません。学習に使った画像と未ラベル画像を区別して観察しましょう。

**観察ミッション**：境界近くの画像を探す → 上の画面でラベルを追加する → AIを更新する → 同じ画像の位置と色を見比べる。ラベルをリセットしても地図の配置は変わりません。


In [ ]:
# 【触らない】観察用の図です。分類器は元の全特徴量で学習します。
import matplotlib.pyplot as plt
from io import BytesIO
from sklearn.manifold import TSNE

# セルの再実行時に、ボタンの処理が重複しないようにします。
for button, callback in globals().get("_map_callbacks", []):
    button.on_click(callback, remove=True)

map_out = widgets.Output()
# 図はOutputの消去・再表示に依存せず、専用の画像ウィジェットで更新します。
map_plots = [widgets.Image(format="png", layout=widgets.Layout(width="100%", max_width="800px")) for _ in range(2)]
map_refresh = widgets.Button(description="図を更新する", icon="refresh")
map_index = widgets.BoundedIntText(value=0, min=0, max=len(metadata)-1, description="画像No.")
map_preview = widgets.Output()
map_coordinates = None
# 縦軸は重なりを減らす固定の散らし位置。特徴や好みを表しません。
score_y = np.random.default_rng(42).uniform(-1, 1, len(metadata))

def draw_maps(_=None):
    global map_coordinates
    with map_out:
        clear_output(wait=True)
        if model is None:
            for plot in map_plots:
                plot.value = b""
            print("先に両方のラベルを付けて『AIを更新する』を押してください。")
            return
        if len(embeddings) < 3 or not np.isfinite(embeddings).all():
            for plot in map_plots:
                plot.value = b""
            print("図には3枚以上の画像と、欠損のない特徴量が必要です。講師に確認してください。")
            return
        if map_coordinates is None:
            print("画像の地図を作っています。初回だけ少し待ってください。")
            map_coordinates = TSNE(
                n_components=2, perplexity=min(30, len(embeddings)-1),
                init="random", learning_rate="auto", random_state=42,
            ).fit_transform(embeddings)
            clear_output(wait=True)
        decision = model.decision_function(embeddings)
        probabilities = model.predict_proba(embeddings)[:, 1]
        print("色: 青緑=今回はちがう寄り / 赤=かわいい！寄り。色はAIの予測です。")
        print("形: ○=未ラベル / △=かわいい！ / □=今回はちがう。形は今のあなたのラベルです。")
        print("地図は近所の似かたを見る図です。島の距離や大きさに意味を付けすぎないでね。")
        print("スコア図の縦位置は重なりを減らすためのものです。上下に意味はありません。")
        print("ラベル追加だけではAIは変わりません。『AIを更新する』で学習し直します。")
        rendered_plots = []
        for title, xy in [
            ("Image map (t-SNE)", map_coordinates),
            ("AI decision score", np.column_stack([decision, score_y])),
        ]:
            fig, ax = plt.subplots(figsize=(8, 4.5), constrained_layout=True)
            if title == "AI decision score":
                bound = max(1.0, float(np.max(np.abs(decision))) * 1.12)
                ax.axvspan(-bound, 0, color="#d9efec", alpha=0.5)
                ax.axvspan(0, bound, color="#fce0e0", alpha=0.5)
                ax.axvline(0, color="black", linestyle="--", label="Boundary (50%)")
                ax.set_xlim(-bound, bound)
                ax.set_xlabel("Decision score: different < 0 < cute")
                ax.set_yticks([])
            else:
                ax.set_xticks([])
                ax.set_yticks([])
            for label, marker, name in [(-1, "o", "Unlabeled"), (1, "^", "Cute"), (0, "s", "Different")]:
                mask = labels == label
                points = ax.scatter(
                    xy[mask, 0], xy[mask, 1], c=probabilities[mask],
                    cmap="RdBu_r", vmin=0, vmax=1, marker=marker,
                    s=45, edgecolors="#333333", linewidths=0.5, label=name,
                )
            selected = map_index.value
            ax.scatter(*xy[selected], s=200, facecolors="none", edgecolors="black", linewidths=2)
            ax.annotate(f"No.{selected}", xy[selected], xytext=(6, 8), textcoords="offset points")
            ax.set_title(title)
            ax.legend(loc="upper left", fontsize=8)
            fig.colorbar(points, ax=ax, label="AI cute score", ticks=[0, 0.5, 1])
            try:
                with BytesIO() as buffer:
                    fig.savefig(buffer, format="png", dpi=110)
                    rendered_plots.append(buffer.getvalue())
            finally:
                plt.close(fig)
        for plot, png in zip(map_plots, rendered_plots):
            plot.value = png

def preview_map_image(_=None):
    with map_preview:
        clear_output(wait=True)
        idx = map_index.value
        try:
            with Image.open(image_path(idx)) as source:
                preview = source.convert("RGB")
            preview.thumbnail((180, 180))
            display(preview)
            print(f"No.{idx} / {metadata.iloc[idx].get('breed', '?')}")
        except OSError:
            print("画像を開けませんでした。imagesフォルダを講師に確認してください。")

def refresh_maps(_=None):
    draw_maps()
    preview_map_image()

_map_callbacks = [(train_button, refresh_maps), (reset_button, refresh_maps),
                  (map_refresh, refresh_maps), (cute_button, refresh_maps),
                  (different_button, refresh_maps)]
for button, callback in _map_callbacks:
    button.on_click(callback)
map_index.observe(refresh_maps, names="value")
display(widgets.VBox([widgets.HBox([map_refresh, map_index]), map_preview, map_out, *map_plots]))
refresh_maps()


## 6-B. 実験: ロジスティック回帰と3層MLPを比べる

【実行してOK・コードは触らない】今までのAIはそのままです。この実験専用の2つのAIを、同じ特徴量・同じラベルで学習します。「比較する」を押した時点の結果なので、ラベルを増やしたらもう一度押してください。

MLPは「入力 → 16個の隠れユニット（ReLU）→ 出力」の3層です。画像の特徴を取り出すEfficientNetは固定したまま、CPUでこの小さな分類器だけ学習します。

- 両方のラベルが1枚以上あれば比較できます。どちらもラベル枚数の偏りを重みで補正します。
- 3-Aで固定した確認用画像を使います。標準化も分類器も学習用画像だけで学習します。
- 確認用は全モデル共通で固定です。少数画像の参考値で、結果を見て繰り返し調整すると確認用への合わせすぎも起こります。
- 推しTOP5は未ラベル画像だけから選びます。評価用ラベルがない画像では、正解率は計算できません。
- 複雑なAIほどよいとは限りません。学習用と確認用の差、自分の好みとの違いを観察しましょう。MLPが必ず過学習するわけではありません。


In [ ]:
# 【触らない】この比較ではメイン画面の model / scores / labels を変更しません。
import warnings
import inspect
from time import perf_counter
from sklearn.neural_network import MLPClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.exceptions import ConvergenceWarning

def fit_mlp_comparison(x, label_snapshot, validation_mask, validation_labels):
    if "sample_weight" not in inspect.signature(MLPClassifier.fit).parameters:
        raise ValueError("この実験にはscikit-learn 1.7以上が必要です。uv sync --locked 後にカーネルを再起動してください。")
    groups = [np.flatnonzero((label_snapshot == value) & ~validation_mask) for value in (0, 1)]
    if any(len(group) == 0 for group in groups):
        raise ValueError("『かわいい！』と『今回はちがう』を最低1枚ずつ教えてください。")
    train_ids = np.flatnonzero((label_snapshot != -1) & ~validation_mask)
    check_ids = np.flatnonzero(validation_mask)
    if not len(check_ids) or np.any(validation_labels[check_ids] < 0):
        raise ValueError("先に確認用画像すべてに回答してください。")
    targets = label_snapshot[train_ids]
    learners = {
        "ロジスティック回帰": LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42),
        "3層MLP": MLPClassifier(hidden_layer_sizes=(16,), activation="relu",
                              solver="lbfgs", alpha=1.0, max_iter=500,
                              max_fun=15000, random_state=42),
    }
    fitted, predictions, rows = {}, {}, []
    for name, learner in learners.items():
        pipeline = Pipeline([("scaler", StandardScaler()), ("clf", learner)])
        kwargs = {}
        if isinstance(learner, MLPClassifier):
            kwargs["clf__sample_weight"] = compute_sample_weight("balanced", targets)
        started = perf_counter()
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always", ConvergenceWarning)
            pipeline.fit(x[train_ids], targets, **kwargs)
        elapsed = perf_counter() - started
        not_converged = any(issubclass(w.category, ConvergenceWarning) for w in caught)
        fitted[name] = pipeline
        predictions[name] = pipeline.predict_proba(x)[:, 1]
        rows.append({
            "モデル": name, "学習枚数": len(train_ids), "確認枚数": len(check_ids),
            "学習用一致率": pipeline.score(x[train_ids], targets),
            "確認用一致率": pipeline.score(x[check_ids], validation_labels[check_ids]) if len(check_ids) else np.nan,
            "学習秒": round(elapsed, 3),
            "学習状態": "収束せず（参考結果）" if not_converged else "終了",
        })
    return {"models": fitted, "scores": predictions, "summary": pd.DataFrame(rows),
            "train_ids": train_ids, "check_ids": check_ids}

mlp_compare_button = widgets.Button(description="比較する", icon="balance-scale")
mlp_compare_out = widgets.Output()

def show_mlp_comparison(_=None):
    mlp_compare_button.disabled = True
    try:
        with mlp_compare_out:
            clear_output(wait=True)
            snapshot = labels.copy()
            try:
                result = fit_mlp_comparison(embeddings, snapshot, holdout_mask, holdout_labels)
                for name, fitted in result["models"].items():
                    record_evaluation(name, fitted, (snapshot != -1) & ~holdout_mask)
            except ValueError as exc:
                print(exc)
                return
            print("比較ボタンを押した時点の結果です。メインのAI・出題・地図は変えていません。")
            display(result["summary"].style.format({
                "学習用一致率": "{:.1%}", "確認用一致率": "{:.1%}", "学習秒": "{:.3f}"
            }, na_rep="未評価"))
            if len(result["check_ids"]):
                print("確認用画像No.:", result["check_ids"].tolist())
                print("確認用の一致率は参考値です。高い予測スコアだけでは過学習とは判断できません。")
            else:
                print("先に確認用画像すべてに回答してください。")
            candidates = np.flatnonzero((snapshot == -1) & ~holdout_mask)
            if len(candidates) == 0:
                print("未ラベル画像がないため、推し比較は省略します。")
                return
            for name, values in result["scores"].items():
                print(f"{name}: 未ラベル画像の推しTOP5")
                for idx in candidates[np.argsort(-values[candidates])[:5]]:
                    with Image.open(image_path(int(idx))) as source:
                        thumbnail = source.convert("RGB")
                    thumbnail.thumbnail((160, 160))
                    display(thumbnail)
                    print(f"No.{idx} / AIかわいい度: {values[idx]*100:.1f}%")
            lr_values, mlp_values = result["scores"].values()
            selected = candidates[np.argsort(-np.abs(lr_values[candidates]-mlp_values[candidates]))[:10]]
            print("予測の違いが大きい未ラベル画像")
            display(pd.DataFrame({
                "画像No.": selected, "ロジスティック回帰": lr_values[selected],
                "3層MLP": mlp_values[selected],
            }).style.format({"ロジスティック回帰": "{:.1%}", "3層MLP": "{:.1%}"}))
    except OSError as exc:
        with mlp_compare_out:
            print(f"画像を開けません。データの場所を確認してください: {exc}")
    finally:
        mlp_compare_button.disabled = False

mlp_compare_button.on_click(show_mlp_comparison)
display(widgets.VBox([mlp_compare_button, mlp_compare_out]))


## 6-C. 学習枚数と一致率の記録

4の画面で10枚→累計20枚→累計40枚と教え、各段階で「この枚数で比較・記録」を押します。「記録を見る」で同じ確認用写真への一致率を比べます。セルを再実行する必要はありません。

出題はランダムにそろえています。確認用の回答はAIには教えません。両方の回答がそろわないときは目標枚数を超えて回答でき、実際の枚数で記録します。

40枚で比較した後は、4の「自由に追加して学習する」で残りの学習用写真に回答できます。「AIを更新する」で追加分を学習し、必要なら「この枚数で比較・記録」で比較も残せます。10・20・40枚の記録と確認用回答はそのままです。


In [ ]:
# 記録の表示もボタンで行います。
record_button = widgets.Button(description="記録を見る（4の画面に表示）", layout=widgets.Layout(width="240px"))
record_button.on_click(show_history)
display(record_button)


## 7. ふりかえり

最後に、次の3つを考えてみましょう。

1. AIの推しTOP10は、自分の好みに近かったですか？
2. 「思い込み強めAI」は、ふつうのAIと比べてどこが違いましたか？
3. AIにもっと自分の好みを教えるなら、次はどんな画像にラベルを付けたいですか？


## 8. 保存

回答・評価履歴は教材フォルダの `participant_progress.json` に自動保存されます。「CSV保存」で提出・分析用の回答と評価履歴も保存できます。

別の参加者に交代するときは、4の「新しい参加者」を押し、確認してからリセットしてください。以前の回答は日時付きJSONにバックアップされます。
